# Matrices TF DF
- **Estudiante**: Jose Luis Andino
- **Fecha de Entrega:** Miercoles 28 de Octubre del 2026
- **Curso:** GR1CC
- **Carrera:** Computación
- **Docente:** Prof. Iván Carrera

In [3]:
import re
import unicodedata
import nltk
import pandas as pd
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# 1. Descarga de recursos (silenciosa)
nltk.download('gutenberg', quiet=True)
nombres_archivos = gutenberg.fileids()

# 2. Configuración de normalización
def normalizar_texto(texto):
    """
    Normaliza el texto pasando a minúsculas, eliminando diacríticos 
    y extrayendo únicamente tokens alfanuméricos.
    """
    texto_norm = unicodedata.normalize("NFKD", texto.lower())
    texto_limpio = texto_norm.encode("ascii", "ignore").decode("ascii")
    tokens = re.findall(r"[a-z0-9]+", texto_limpio)
    return " ".join(tokens)

print("Procesando y normalizando documentos...")
documentos = [normalizar_texto(gutenberg.raw(nombre)) for nombre in nombres_archivos]

# ==========================================
# PASO 1: Creación de la Matriz TF
# ==========================================
print("\n--- 1. Matriz TF ---")
vectorizador_tf = CountVectorizer()
matriz_tf = vectorizador_tf.fit_transform(documentos)
vocabulario = vectorizador_tf.get_feature_names_out()

print(f"Dimensiones Matriz TF: {matriz_tf.shape[0]} documentos x {matriz_tf.shape[1]} términos únicos")

# ==========================================
# PASO 2 y 3: Cálculo de IDF y Matriz TF-IDF
# ==========================================
print("\n--- 2. Vector IDF y Matriz TF-IDF ---")
transformador_tfidf = TfidfTransformer(smooth_idf=True)
matriz_tfidf = transformador_tfidf.fit_transform(matriz_tf)
vector_idf = transformador_tfidf.idf_

# ==========================================
# VISUALIZACIÓN DE RESULTADOS
# ==========================================
terminos_interes = ['the', 'whale', 'ahab', 'caesar', 'sea']
terminos_validos = [t for t in terminos_interes if t in vocabulario]

df_tfidf = pd.DataFrame(matriz_tfidf.toarray(), index=nombres_archivos, columns=vocabulario)

print("\nFrecuencia Inversa de Documento (IDF) para términos clave:")
for term in terminos_validos:
    indice_term = list(vocabulario).index(term)
    print(f"'{term}': {vector_idf[indice_term]:.4f}")

print("\nMatriz TF-IDF (Muestra para 5 textos y términos específicos):")
textos_muestra = ['melville-moby_dick.txt', 'shakespeare-caesar.txt', 'bible-kjv.txt', 'austen-emma.txt']
print(df_tfidf.loc[textos_muestra, terminos_validos])

Procesando y normalizando documentos...

--- 1. Matriz TF ---
Dimensiones Matriz TF: 18 documentos x 41803 términos únicos

--- 2. Vector IDF y Matriz TF-IDF ---

Frecuencia Inversa de Documento (IDF) para términos clave:
'the': 1.0000
'whale': 1.9985
'ahab': 2.8458
'caesar': 1.7472
'sea': 1.1112

Matriz TF-IDF (Muestra para 5 textos y términos específicos):
                             the     whale      ahab    caesar       sea
melville-moby_dick.txt  0.695608  0.118105  0.070097  0.000168  0.024371
shakespeare-caesar.txt  0.305468  0.000000  0.000000  0.175141  0.001173
bible-kjv.txt           0.619350  0.000058  0.002588  0.000507  0.004300
austen-emma.txt         0.360996  0.000000  0.000000  0.000000  0.001079
